## Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

print("Bibliotheques importees.")

Bibliotheques importees.


## Parametres

In [3]:
# Chemin du fichier source et de la colonne cible
INPUT_FILE  = "../../data/automobile/imports-85.data"
TARGET_COL  = "price"
RANDOM_STATE = 42     # graine fixe pour la reproductibilite du split
TEST_SIZE    = 0.2    # 80% train / 20% test

# Dossier de sortie ou seront sauvegardes le modele et les splits
OUTPUT_DIR = "../../outputs/automobile"

## Chargement et nettoyage des données

In [4]:
# Noms des 26 colonnes du dataset automobile (dataset brut du UCI Repository)
col_names = [
    "symboling", "normalized_losses", "make", "fuel_type", "aspiration",
    "num_doors", "body_style", "drive_wheels", "engine_location",
    "wheel_base", "length", "width", "height", "curb_weight",
    "engine_type", "num_cylinders", "engine_size", "fuel_system",
    "bore", "stroke", "compression_ratio", "horsepower", "peak_rpm",
    "city_mpg", "highway_mpg", "price"
]

print("[SVM] Chargement du fichier source...")
data = pd.read_csv(INPUT_FILE, sep=",", names=col_names, na_values="?")

# On retire la colonne symboling (non pertinente) et toutes les colonnes
# categorielles : on ne garde que les attributs numeriques + la cible
data = data.drop(columns=["symboling"])
data = data.select_dtypes(include=[np.number])
data = data.dropna().reset_index(drop=True)

print(f"[SVM] Dimensions apres nettoyage : {data.shape}")
data.to_csv(f"{OUTPUT_DIR}/automobile_data.csv", index=False, sep=";")
data.head()

[SVM] Chargement du fichier source...
[SVM] Dimensions apres nettoyage : (160, 15)


,normalized_losses,wheel_base,length,width,height,curb_weight,engine_size,bore,stroke,compression_ratio,horsepower,peak_rpm,city_mpg,highway_mpg,price
0,164.0,99.8,176.6,66.2,54.3,2337,109,3.19,3.4,10.0,102.0,5500.0,24,30,13950.0
1,164.0,99.4,176.6,66.4,54.3,2824,136,3.19,3.4,8.0,115.0,5500.0,18,22,17450.0
2,158.0,105.8,192.7,71.4,55.7,2844,136,3.19,3.4,8.5,110.0,5500.0,19,25,17710.0
3,158.0,105.8,192.7,71.4,55.9,3086,131,3.13,3.4,8.3,140.0,5500.0,17,20,23875.0
4,192.0,101.2,176.8,64.8,54.3,2395,108,3.50,2.8,8.8,101.0,5800.0,23,29,16430.0


## Séparation train/test (80/20)

In [ ]:
# Le split train/test est celui qui servira a la fois :
#  - a la Phase 1 (motifs graduels extraits UNIQUEMENT sur X_train)
#  - a la Phase 3 (contrefactuels generes sur des instances de X_test)
X = data.drop(columns=[TARGET_COL])
y = data[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print(f"[SVM] Train : {X_train.shape[0]} lignes | Test : {X_test.shape[0]} lignes")

# Sauvegarde des splits pour reutilisation dans les notebooks 2 et 3
X_train.to_csv(f"{OUTPUT_DIR}/X_train.csv", index=False, sep=";")
X_test.to_csv(f"{OUTPUT_DIR}/X_test.csv", index=False, sep=";")
y_train.to_csv(f"{OUTPUT_DIR}/y_train.csv", index=False, sep=";")
y_test.to_csv(f"{OUTPUT_DIR}/y_test.csv", index=False, sep=";")
print("[SVM] Splits sauvegardes dans outputs/automobile/")

[SVM] Train : 128 lignes | Test : 32 lignes
[SVM] Splits sauvegardes dans outputs/automobile/


## Entraînement du SVM

In [ ]:
# Le SVM (SVR) est tres sensible a l'echelle des variables alors on standardise donc :
#  - les features X, via un StandardScaler a l'interieur d'un pipeline
#  - la cible y, via TransformedTargetRegressor (qui standardise y avant
#    l'entrainement et "de-standardise" automatiquement les predictions)
# Ainsi, model.predict(x) prend et renvoie des valeurs dans l'echelle REELLE :
# les Phases 1 et 3 n'ont pas a se soucier de cette mise a l'echelle.
print("[SVM] Entrainement du modele SVR (noyau RBF, hyperparametres par defaut)...")

base_pipeline = make_pipeline(StandardScaler(), SVR(kernel="rbf"))
model = TransformedTargetRegressor(regressor=base_pipeline, transformer=StandardScaler())
model.fit(X_train, y_train)

print("[SVM] Entrainement termine.")

[SVM] Entrainement du modele SVR (noyau RBF, hyperparametres par defaut)...
[SVM] Entrainement termine.


## Évaluation

In [ ]:
print("[SVM] Evaluation sur l'ensemble de test ...")

y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print(f"[SVM] RMSE : {rmse:.2f}")
print(f"[SVM] MAE  : {mae:.2f}")
print(f"[SVM] R2   : {r2:.4f}")
print(f"[SVM] MAPE : {mape:.2f}%")

pd.DataFrame([{"RMSE": rmse, "MAE": mae, "R2": r2, "MAPE": mape}]).to_csv(
    f"{OUTPUT_DIR}/metriques_svm.csv", index=False, sep=";"
)

[SVM] Evaluation sur l'ensemble de test (holdout)...
[SVM] RMSE : 1539.85
[SVM] MAE  : 1093.70
[SVM] R2   : 0.8177
[SVM] MAPE : 11.74%


## Sauvegarde du modèle

In [8]:
joblib.dump(model, f"{OUTPUT_DIR}/svm_model.pkl")
print(f"[SVM] Modele sauvegarde dans {OUTPUT_DIR}/svm_model.pkl")
print("[SVM] Ce notebook n'a besoin d'etre execute qu'UNE SEULE FOIS.")
print("[SVM] Les notebooks 2 et 3 reutiliseront directement ce modele et ces splits.")

[SVM] Modele sauvegarde dans ../../outputs/automobile/svm_model.pkl
[SVM] Ce notebook n'a besoin d'etre execute qu'UNE SEULE FOIS.
[SVM] Les notebooks 2 et 3 reutiliseront directement ce modele et ces splits.
